# 08 -- Building modal

Transfer functions, coherence and mode shapes from the vertical array (windowed signals).

In [1]:
%matplotlib inline

import numpy as np
from asdea_sensors import SensorDataset
from asdea_sensors.config import settings

# The example data folder (edit to your path if needed).
DATA = r"C:\Users\ppala\Desktop\02_31MAY2026"

# A short window keeps every example fast (one ~10 min file).
WSTART, WLEN = "2026-05-31 15:00:00", "10min"

In [2]:
ds = SensorDataset(DATA, verbose=True)
ds.devices

------------------------------------------------------------
SensorDataset
------------------------------------------------------------
path        : C:\Users\ppala\Desktop\02_31MAY2026
files       : 32
time span   : 2026-05-31 14:52:12  ->  2026-05-31 20:02:13
duration    : 18601.0 s
devices     : MNAT0031, MNAT0034, MOF00134, MOF00135, MOF00136
fs / dt     : 252.5885 Hz / 0.003959 s
------------------------------------------------------------
axes (per sensor):
  MNAT0031   -> (3, 1, 5)
  MNAT0034   -> (3, 1, 5)
  MOF00134   -> (0, 1, 2)
  MOF00135   -> (3, 1, 5)
  MOF00136   -> (3, 1, 5)
------------------------------------------------------------
on-disk size: 782.17 MB
RAM         : used 31.66 GB / avail 1.85 GB (94%)
------------------------------------------------------------


['MNAT0031', 'MNAT0034', 'MOF00134', 'MOF00135', 'MOF00136']

In [3]:
# Sensor geometry up front (approximate UTM E/N, elevation). Building methods
# (torsion, mode shapes, drift) need these references.
settings.SENSOR_GEOMETRY["MOF00134"].update(E=500000.0, N=6300000.0, elev=0.0)
settings.SENSOR_GEOMETRY["MNAT0031"].update(E=500000.0, N=6300000.0, elev=7.0)
settings.SENSOR_GEOMETRY["MNAT0034"].update(E=500000.0, N=6300000.0, elev=10.5)
settings.SENSOR_GEOMETRY["MOF00135"].update(E=500000.0, N=6300000.0, elev=14.0)
settings.SENSOR_GEOMETRY["MOF00136"].update(E=500006.0, N=6300004.0, elev=14.0)
{d: settings.SENSOR_GEOMETRY[d]["floor"] for d in settings.SENSOR_GEOMETRY}

{'MOF00134': -1, 'MNAT0031': 2, 'MNAT0034': 3, 'MOF00135': 4, 'MOF00136': 4}

## Read windowed, filtered signals along the array

In [4]:
def comp(dev):
    return ds.device(dev).window(WSTART, WLEN).filter(0.1, 24.9).signal(components="x")
sigs = {d: comp(d) for d in ["MOF00134", "MNAT0031", "MNAT0034", "MOF00135"]}
dt = sigs["MOF00134"].dt
print("read floors:", list(sigs.keys()))

[signal] MOF00134 n=149340 dt=0.004018 comps=x


C:\Dropbox\01. Brain\11. GitHub\ladruno_ASDEA_sensors\src\asdea_sensors\derive\filters.py:49: UserWarning: obspy not available; falling back to the scipy engine
  warnings.warn("obspy not available; falling back to the scipy engine")
C:\Dropbox\01. Brain\11. GitHub\ladruno_ASDEA_sensors\src\asdea_sensors\core\h5_reader.py:45: UserWarning: H5Reader: axes (3, 1, 5) for 'MNAT0031' exceed the 3 available columns; falling back to (0, 1, 2)
  warnings.warn(
C:\Dropbox\01. Brain\11. GitHub\ladruno_ASDEA_sensors\src\asdea_sensors\core\h5_reader.py:45: UserWarning: H5Reader: axes (3, 1, 5) for 'MNAT0034' exceed the 3 available columns; falling back to (0, 1, 2)
  warnings.warn(


[signal] MNAT0031 n=151560 dt=0.003959 comps=x
[signal] MNAT0034 n=151320 dt=0.003966 comps=x


[signal] MOF00135 n=149240 dt=0.004021 comps=x
read floors: ['MOF00134', 'MNAT0031', 'MNAT0034', 'MOF00135']


C:\Dropbox\01. Brain\11. GitHub\ladruno_ASDEA_sensors\src\asdea_sensors\core\h5_reader.py:45: UserWarning: H5Reader: axes (3, 1, 5) for 'MOF00135' exceed the 3 available columns; falling back to (0, 1, 2)
  warnings.warn(


## Transfer function base -> floor 4

In [5]:
from asdea_sensors.building import transfer_function, coherence, modal
frf = transfer_function.compute(sigs["MOF00135"].acc_x, sigs["MOF00134"].acc_x, dt, smooth="konno", fmax=25.0)
print("modal freqs (Hz):", [round(float(x),2) for x in frf["modal_freqs"][:5]])

modal freqs (Hz): [0.24, 0.97, 1.46, 2.43, 3.16]


## Coherence

In [6]:
coh = coherence.compute(sigs["MOF00135"].acc_x, sigs["MOF00134"].acc_x, dt)
print("coherence points:", len(coh["f"]))

coherence points: 513


## Mode shapes across the array

In [7]:
signal_map = {d: sigs[d].acc_x for d in sigs}
shapes = modal.mode_shapes(signal_map, settings.SENSOR_GEOMETRY, dt, component="x", n_modes=2)
print("modal freqs:", [round(float(x),2) for x in np.atleast_1d(shapes["freqs"])[:2]])

modal freqs: [0.51, 0.69]
